# Chapter 04 — The Context Window Is a Budget

**Companion to *Pi Agents*** · [`Pi Agents` chapter 04](https://programmer.ie/books/pi/04-chapter/)

| | |
|---|---|
| Chapter | `04-chapter.md` · weight 4 · `/books/pi/04-chapter/` |
| Notebook | `notebooks/pi/04-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Is compaction an opinion or arithmetic? **Find the exact turn at which
Pi decides to compact** and check it against the one-line rule the chapter states.

The rule is `contextTokens > contextWindow - reserveTokens`. If that is what runs,
the boundary is inspectable — you can call the decision function yourself at every
token count and find where it flips.


## What this notebook establishes

- The trigger is arithmetic and its boundary is exact: Pi compacts when `contextTokens > contextWindow - reserveTokens`, and not one turn earlier. The exported `shouldCompact` is the function that decides, and the cell below walks it across the boundary.
- A larger `reserveTokens` compacts earlier on the *same* conversation. It is not a preference — it moves the turn.
- The percentage in the footer is `tokens / contextWindow`, checked against the real `getContextUsage()` of a real session.
- Only **active** tools are declared to the model. Narrowing the set with `tools:` reduces what the model is told *and* what is registered; the registered-but-not-active gap comes from `deferred` exposure, which is chapter 17.

## What this notebook does **not** establish

- **Token counts come from the faux provider's scripted usage.** They are real numbers produced by a real agent, but a real model's usage would differ and the *turn* at which compaction fires would differ with it.
- **Nothing about summary quality.** Compaction here is about the cut point and the arithmetic; chapter 23 is about what is kept.
- **Nothing about provider context-overflow recovery**, which is a repair path rather than a strategy — chapter 29.
- **No guidance for choosing `reserveTokens`.** `metadata/04-chapter.yaml` records that the chapter gives none, and neither does this notebook.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — Pi's exported `shouldCompact` across its boundary, a real session's context accounting, and the chapter's own tests.
* **Evidence scope:** `in_process_runtime`, plus a `declarations` reading of the exported signature. The provider is scripted.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(4))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/04-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the decision function, read from the pin

`shouldCompact` is exported from `@earendil-works/pi-coding-agent` at the pin. Its
declaration is the contract; calling it is how you find the boundary rather than
inferring it from a log.

In [3]:
DRIVER = r"""
import { shouldCompact } from "@earendil-works/pi-coding-agent";

const WINDOW = 100_000;
const RESERVE = 16_384;
const settings = { enabled: true, reserveTokens: RESERVE, keepRecentTokens: 8_000 };
const threshold = WINDOW - RESERVE;

const around = [threshold - 2, threshold - 1, threshold, threshold + 1, threshold + 2];

console.log(JSON.stringify({
  window: WINDOW,
  reserveTokens: RESERVE,
  threshold,
  aroundThreshold: around.map((t) => ({ tokens: t, compacts: shouldCompact(t, WINDOW, settings as any) })),
  reserveMovesTheLine: [16_384, 32_000, 64_000].map((r) => ({ reserveTokens: r, firstTokenThatCompacts: WINDOW - r + 1 })),
  disabled: [threshold, threshold + 10_000].map((t) => ({ tokens: t, compacts: shouldCompact(t, WINDOW, { ...settings, enabled: false } as any) })),
}));
"""
boundary = pinb.driver(DRIVER, label="ch04-should-compact", timeout=120)
print(f"contextWindow {boundary['window']:,}   reserveTokens {boundary['reserveTokens']:,}   "
      f"threshold {boundary['threshold']:,}")
print()
print(pinb.table(["contextTokens", "compacts?"], [[f"{row['tokens']:,}", "yes" if row["compacts"] else "no"] for row in boundary["aroundThreshold"]], indent=""))
print()
print(pinb.table(["reserveTokens", "first token count that compacts"],
                 [[f"{row['reserveTokens']:,}", f"{row['firstTokenThatCompacts']:,}"] for row in boundary["reserveMovesTheLine"]], indent=""))

contextWindow 100,000   reserveTokens 16,384   threshold 83,616

contextTokens  compacts?
-------------  ---------
83,614  no
83,615  no
83,616  no
83,617  yes
83,618  yes

reserveTokens  first token count that compacts
-------------  -------------------------------
16,384  83,617
32,000  68,001
64,000  36,001


The flip is exactly where the chapter says it is: `threshold` does not compact,
`threshold + 1` does. There is no tolerance band and no hysteresis at this layer.

The second table is the practical consequence. `reserveTokens` is not slack for
being wrong — it is the distance between *the point of no return* and *the point at
which you compact*, and doubling it doubles how early compaction happens.

In [4]:
t = boundary["threshold"]

pinb.show_checks([
    pinb.check("threshold - 1 does not compact", not boundary["aroundThreshold"][1]["compacts"], "no"),
    pinb.check("threshold itself does not compact", not boundary["aroundThreshold"][2]["compacts"], "the comparison is strict >"),
    pinb.check("threshold + 1 does compact", boundary["aroundThreshold"][3]["compacts"], "yes"),
    pinb.check("threshold == window - reserve", t == boundary["window"] - boundary["reserveTokens"], str(t)),
    pinb.check("enabled: false compacts at no token count",
               not any(r["compacts"] for r in boundary["disabled"]), str(boundary["disabled"])),
    pinb.check("a larger reserve moves the line earlier",
               [r["firstTokenThatCompacts"] for r in boundary["reserveMovesTheLine"]] == sorted(
                   [r["firstTokenThatCompacts"] for r in boundary["reserveMovesTheLine"]], reverse=True),
               "monotonically earlier"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------  --------------------------------------------------------------------------
PASS  threshold - 1 does not compact  no
PASS  threshold itself does not compact  the comparison is strict >
PASS  threshold + 1 does compact  yes
PASS  threshold == window - reserve  83616
PASS  enabled: false compacts at no token count  [{'tokens': 83616, 'compacts': False}, {'tokens': 93616, 'compacts': Fals…
PASS  a larger reserve moves the line earlier  monotonically earlier

6/6 assertions held.


## Experiment: the same conversation, real context accounting

Now the arithmetic against a *real* session. A short conversation is run with a
scripted model, and the session's own `getContextUsage()` is read after each turn.

Two things are being checked: that the percent really is tokens over window, and
that only **active** tools are declared to the model — the other boundary this
chapter draws.

In [5]:
DRIVER2 = r"""
import { fauxAssistantMessage, fauxText } from "@earendil-works/pi-ai";
import { makeSession } from "../harness/session.ts";
import { pad } from "../harness/history.ts";

const say = (t: string) => fauxAssistantMessage([fauxText(t)]);

async function conversation(tools?: string[]) {
	const h = await makeSession({ extensions: [], tools });
	const turns: any[] = [];
	let declared: string[] | null = null;
	for (let i = 0; i < 5; i++) {
		const answer = say(pad(`answer ${i}`, 10 * (i + 1)));
		if (declared === null) {
			h.faux.appendResponses([
				(ctx: any) => {
					const sys: any = ctx.messages.find((m: any) => m.role === 'system');
					declared = (sys?.toolsAdded ?? []).map((t: any) => t.name);
					return answer;
				},
			]);
		}
		else {
			h.faux.setResponses([answer]);
		}
		await h.session.prompt(`question ${i}`);
		const u = h.session.getContextUsage();
		turns.push({ turn: i + 1, tokens: u?.tokens ?? null, contextWindow: u?.contextWindow ?? null, percent: u?.percent ?? null, messages: h.session.messages.length });
	}
	const active = h.session.getActiveToolNames();
	const registered = h.session.getAllTools().map((t: any) => t.name);
	h.dispose();
	return { turns, declared, active, registered };
}

console.log(JSON.stringify({ default: await conversation(), narrowed: await conversation(['read']) }));
"""
usage = pinb.driver(DRIVER2, label="ch04-context-usage", timeout=240)
print("A default session, five turns (each answer padded, so the conversation really grows):")
print(pinb.table(["turn", "messages", "context tokens", "contextWindow", "percent"],
                 [[t["turn"], t["messages"], f"{t['tokens']:,}" if t["tokens"] else "-", f"{t['contextWindow']:,}",
                   f"{t['percent']:.2f}" if t["percent"] is not None else "-"]
                  for t in usage["default"]["turns"]], indent=""))
print()
print(pinb.md_table(
    ["session", "tools declared to the model", "active", "registered"],
    [["default", ", ".join(usage["default"]["declared"]), ", ".join(usage["default"]["active"]), len(usage["default"]["registered"])],
     ["tools: [read]", ", ".join(usage["narrowed"]["declared"]), ", ".join(usage["narrowed"]["active"]), len(usage["narrowed"]["registered"])]],
))

A default session, five turns (each answer padded, so the conversation really grows):
turn  messages  context tokens  contextWindow  percent
----  --------  --------------  -------------  -------
1  3  3,095  100,000  3.09
2  5  1,848  100,000  1.85
3  7  2,178  100,000  2.18
4  9  2,588  100,000  2.59
5  11  3,078  100,000  3.08

| session | tools declared to the model | active | registered |
|---|---|---|---|
| default | read, bash, edit, write | read, bash, edit, write | 8 |
| tools: [read] | read | read | 1 |


In [6]:
d = usage["default"]
n = usage["narrowed"]
pcts_ok = all(
    t["percent"] is not None and abs(t["percent"] - (t["tokens"] / t["contextWindow"]) * 100) < 0.01
    for t in d["turns"] if t["tokens"]
)
grew = [t["tokens"] for t in d["turns"] if t["tokens"]]
from_turn_2 = grew[1:]

pinb.show_checks([
    pinb.check("the window is the faux model's declared 100,000",
               all(t["contextWindow"] == 100_000 for t in d["turns"]), "100000"),
    pinb.check("percent is exactly tokens / window", pcts_ok, "within 0.01 of the ratio"),
    pinb.check("context grows strictly from turn 2 on",
               from_turn_2 == sorted(from_turn_2) and len(set(from_turn_2)) == len(from_turn_2),
               f"{from_turn_2[0]:,} -> {from_turn_2[-1]:,} over {len(from_turn_2)} turns"),
    pinb.check("a default session declares bash to the model", "bash" in d["declared"], ", ".join(d["declared"])),
    pinb.check("tools: [read] declares exactly one tool", n["declared"] == ["read"], ", ".join(n["declared"])),
    pinb.check("the tools option narrows registration as well as declaration",
               len(n["registered"]) == len(n["active"]) == 1,
               f"registered {len(n['registered'])}, active {len(n['active'])}"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------------------  ---------------------------
PASS  the window is the faux model's declared 100,000  100000
PASS  percent is exactly tokens / window  within 0.01 of the ratio
PASS  context grows strictly from turn 2 on  1,848 -> 3,078 over 4 turns
PASS  a default session declares bash to the model  read, bash, edit, write
PASS  tools: [read] declares exactly one tool  read
PASS  the tools option narrows registration as well as declaration  registered 1, active 1

6/6 assertions held.


## Contrasting case: `enabled: false` is not "never"

The boundary function refuses at every token count when `enabled: false`. But the
chapter's test asserts something sharper: the **manual** `/compact` still works
with automatic compaction off. Automatic triggering and the ability to compact are
two different switches, and only one of them is a setting.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch04-budget/budget.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch04-budget/budget.test.ts', show="compaction")

$ node --test ch04-budget/budget.test.ts
  PASS  8 passed, 0 failed, 0 skipped  in 1602 ms

  tests matching 'compaction':
    ok   the trigger is arithmetic: compaction happens when contextTokens exceeds contextWindow minus reserveTokens, and not before
    ok   enabled: false turns off automatic compaction, and the manual command still works


The second table is the other half of the chapter's tool boundary, and it is
worth reading carefully: passing `tools: ["read"]` reduced **both** the tools the
model is told about *and* the registered set. The registered-but-not-active gap
this book keeps returning to is not created by narrowing at startup; it is created
by `deferred` exposure (chapter 17), or by an extension that registers a tool and
then sets the active set itself.

## Your turn: predict, then run

**Predict first.** Set `reserveTokens` above the window. At what token count does
the line move, and does `shouldCompact` ever return true below the window?

**Then change one input.** `RESERVE` below feeds the same boundary walk. Try
`100_000` (equal to the window) and `120_000` (larger than it).

**Predict the practical effect.** With `reserveTokens: 100_000` on a 100,000-token
window, when would compaction fire in a real conversation?

In [8]:
RESERVE = 100_000

EXERCISE = DRIVER.replace("const RESERVE = 16_384;", "const RESERVE = Number(process.env.NB_RESERVE);")
edge = pinb.driver(EXERCISE, label="ch04-edge-reserve", timeout=120, env={"NB_RESERVE": str(RESERVE)})
print(f"reserveTokens = {RESERVE:,}  on a {edge['window']:,}-token window")
print(f"threshold = {edge['threshold']:,}")
print()
print(pinb.table(["contextTokens", "compacts?"],
                 [[f"{row['tokens']:,}", "yes" if row["compacts"] else "no"] for row in edge["aroundThreshold"]], indent=""))

print()
if edge["threshold"] <= 0:
    print("A non-positive threshold means the window is already over budget at zero tokens:")
    print("every non-negative token count compacts, so a conversation can never start.")
else:
    print(f"The line sits {edge['threshold']:,} tokens below the window, so compaction is {edge['window'] - edge['threshold']:,} tokens away.")

reserveTokens = 100,000  on a 100,000-token window
threshold = 0

contextTokens  compacts?
-------------  ---------
-2  no
-1  no
0  no
1  yes
2  yes

A non-positive threshold means the window is already over budget at zero tokens:
every non-negative token count compacts, so a conversation can never start.


## Interpretation: a budget with one moving part

The context window is not a limit you hit; it is a threshold you choose, and three
numbers decide it:

```
compact when:   contextTokens  >  contextWindow  -  reserveTokens
                      what Pi sends     the model's ceiling      your choice
```

The two consequences worth carrying forward:

1. **Compaction is early by design.** `reserveTokens` exists so there is room to
   summarise *and* to answer. It is not a safety margin against the provider
   rejecting the request — that is what `keepRecentTokens` and the overflow repair
   path are for (chapter 29).
2. **Compaction does not shrink the system prompt or the tool declarations.** The
   chat turns get smaller; the instructions do not. A long tool-heavy session can
   therefore approach the threshold faster than a long chat session, which the
   chapter states and this notebook does not measure.

### What this chapter does not settle

`metadata/04-chapter.yaml` records two limits that survive here: there is no
guidance for choosing `reserveTokens` or `keepRecentTokens` when the window is
unknown or a provider misreports it, and token estimation *before* the cut point
is chosen is undocumented. Both are honest gaps in the documentation, not things
this notebook can fill.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the exported `shouldCompact` across its boundary, a real session's `getContextUsage()`, and the chapter's own test file.

**Evidence scope:** `in_process_runtime` for the sessions; the boundary walk is a direct call into the pinned package's own decision function.

### What was read or run

- **Ran** `shouldCompact` from `@earendil-works/pi-coding-agent` at the pin, across `threshold - 2 .. threshold + 2`, with `enabled` both true and false.
- **Ran** two real `AgentSession`s (default tools, and `tools: [read]`), reading `getContextUsage()` after each of five turns and capturing the system message the provider received.
- **Ran** `ch04-budget/budget.test.ts` (8 tests), which is where the multi-turn growth runs, the per-model override and the manual `/compact` case.
- **Read** `examples/evidence.json`, chapter 4 rows (8 claims).

### Limitations

- **Token numbers are the faux provider's.** They are produced by a real agent through a real budget, but the growth rate is a property of the scripted answers, not of a real model. The *turn* at which compaction fires would differ with a real conversation.
- **The percentage identity is a definition, not a discovery.** `percent == tokens / window` is what the footer computes; asserting it shows the footer is not lying, not that the token estimate is accurate.
- **No summary quality is assessed.** The boundary is the subject; what compaction keeps is chapter 23.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
